# Battery Management System (BMS) Optimization & Safety Fault Detection
### Comprehensive Presentation & Demonstration Notebook

**Agenda:**
1. **Part 1: EV Fast-Charging Thermal Runaway Benchmark (1,000 samples, 17 sensors)**
   - Data Ingestion & Analytics
   - Step-by-Step Preprocessing, Encoding & Stratified Splitting
   - Literature Baselines: BPSO, BWOA, BHPWOA
   - Proposed HGWO-PSO Co-Tuning Framework
   - Interactive Visualizations (Convergence & SHAP Explainability)
   - Performance Comparison Summary Table
2. **Part 2: NASA Li-ion Classic 4-Cell Benchmark (636 cycles, 24°C)**
   - Data Ingestion & Preprocessing
   - Algorithm Evaluations & Benchmark Comparison
3. **Part 3: NASA Global Multi-Temperature Archive (2,750 cycles, 34 cells, 4°C-44°C)**
   - Multi-Temperature Extreme Robustness Testing
   - Algorithm Evaluations & Benchmark Comparison
4. **Part 4: Cross-Dataset Multi-Panel Summary Visualizations**

In [ ]:
# Step 1: Clone Repository & Setup Environment
import os
if not os.path.exists("BMS"):
    !git clone https://github.com/abdullahgni/BMS.git
    %cd BMS
else:
    %cd BMS
    !git pull origin main

!pip install -q -r requirements.txt

---
## PART 1: EV Fast-Charging Thermal Runaway Telemetry

In [ ]:
# Step 2: Ingest EV Telemetry Data & View Exploratory Analytics
import pandas as pd
import numpy as np

df_ev = pd.read_csv("EV_Battery_Charging_TR_Dataset_with_Notes.csv")
print("Dataset Shape:", df_ev.shape)
print("
--- First 5 Telemetry Records ---")
display(df_ev.head())
print("
--- Telemetry Columns & Data Types ---")
df_ev.info()
print("
--- Statistical Distribution Summary ---")
display(df_ev.describe())

In [ ]:
# Step 3: Telemetry Preprocessing, Encoding & Stratified Split Setup
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold

# 1. Encode discrete protocol features
le = LabelEncoder()
df_ev["ChargingStage"] = le.fit_transform(df_ev["ChargingStage"])
df_ev["MoistureDetected"] = df_ev["MoistureDetected"].astype(int)

# 2. Map ground-truth physical safety targets:
# 0: Critical (Runaway), 1: OK (Watch), 2: Warning (Alarm/Warning)
target_map = {"Runaway": 0, "Watch": 1, "Alarm": 2, "Warning": 2}
y_ev = df_ev["EventFlag"].map(target_map).values

# 3. Discard non-sensor metadata columns
metadata_cols = ["Timestamp", "ChargerID", "CellID", "Notes", "BMS_Status", "EventFlag", "TR_Probability"]
X_ev = df_ev.drop(columns=metadata_cols)
feature_names_ev = X_ev.columns.tolist()

# 4. Standardize physical sensor channels (Z-Score scaling)
scaler_ev = StandardScaler()
X_ev_scaled = pd.DataFrame(scaler_ev.fit_transform(X_ev), columns=feature_names_ev)

# 5. Initialize Stratified 5-Fold Cross-Validation splitter
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Preprocessed Features ({len(feature_names_ev)}): {feature_names_ev}")
print(f"Target Breakdown: Critical (0) = {(y_ev == 0).sum()}, OK (1) = {(y_ev == 1).sum()}, Warning (2) = {(y_ev == 2).sum()}")
display(X_ev_scaled.head())

In [ ]:
# Step 4: Run Baseline 1 - Binary Particle Swarm Optimization (BPSO)
!python bpso.py --dataset ev

In [ ]:
# Step 5: Run Baseline 2 - Binary Whale Optimization Algorithm (BWOA)
!python bwoa.py --dataset ev

In [ ]:
# Step 6: Run Baseline 3 - Binary Hybrid PSO-WOA (BHPWOA)
!python bhpwoa.py --dataset ev

In [ ]:
# Step 7: Run Proposed Framework - HGWO-PSO with Hyperparameter Co-Tuning
!python hgwo_pso.py --dataset ev

In [ ]:
# Step 8: Multi-Objective Convergence Curves (Direct Display)
!python plot_convergence.py

In [ ]:
# Step 9: SHAP Explainable AI (XAI) Feature Attribution (Direct Display)
!python plot_shap.py

In [ ]:
# Step 10: Final Comparison Benchmark Table (EV Dataset)
!python benchmark_comparison.py --dataset ev

---
## PART 2: NASA Li-ion Classic 4-Cell Benchmark (636 cycles at 24°C)
Evaluating cross-dataset transferability on standard experimental battery cells (B0005, B0006, B0007, B0018).

In [ ]:
# Step 11: Ingest NASA Classic 4 Dataset & Preprocess
df_n4 = pd.read_csv("nasa_battery_benchmark_classic4.csv")
print("NASA Classic 4 Shape:", df_n4.shape)
display(df_n4.head())

# Preprocessing
non_features_n4 = ["battery_id", "cycle_file", "Capacity_Ah", "Health_State", "Target_Label"]
feat_n4 = [c for c in df_n4.columns if c not in non_features_n4]
y_n4 = df_n4["Target_Label"].values

scaler_n4 = StandardScaler()
X_n4_scaled = pd.DataFrame(scaler_n4.fit_transform(df_n4[feat_n4]), columns=feat_n4)
print(f"NASA Classic Candidate Features ({len(feat_n4)}): {feat_n4}")
print(f"Target Breakdown: Critical (0) = {(y_n4 == 0).sum()}, OK (1) = {(y_n4 == 1).sum()}, Warning (2) = {(y_n4 == 2).sum()}")

In [ ]:
# Step 12: Run Algorithms on NASA Classic 4
!python bpso.py --dataset nasa
!python bwoa.py --dataset nasa
!python hgwo_pso.py --dataset nasa

In [ ]:
# Step 13: NASA Classic 4 Comparison Table
!python benchmark_comparison.py --dataset nasa

---
## PART 3: NASA Global Multi-Temperature Archive (2,750 cycles, 34 cells, 4°C-44°C)
Evaluating environmental robustness across sub-freezing winter chambers (4°C), room temperature (24°C), and summer heat stress (44°C).

In [ ]:
# Step 14: Ingest NASA Global Archive & Preprocess
df_ng = pd.read_csv("nasa_battery_benchmark_global_2750.csv")
print("NASA Global Archive Shape:", df_ng.shape)
display(df_ng.head())
df_ng.info()

# Preprocessing
non_features_ng = ["battery_id", "cycle_file", "Capacity_Ah", "Health_State", "Target_Label"]
feat_ng = [c for c in df_ng.columns if c not in non_features_ng]
y_ng = df_ng["Target_Label"].values

scaler_ng = StandardScaler()
X_ng_scaled = pd.DataFrame(scaler_ng.fit_transform(df_ng[feat_ng]), columns=feat_ng)
print(f"NASA Global Candidate Features ({len(feat_ng)}): {feat_ng}")
print(f"Target Breakdown: Critical (0) = {(y_ng == 0).sum()}, OK (1) = {(y_ng == 1).sum()}, Warning (2) = {(y_ng == 2).sum()}")

In [ ]:
# Step 15: Run Algorithms on NASA Global Multi-Temperature Archive
!python bpso.py --dataset nasa_global
!python bwoa.py --dataset nasa_global
!python hgwo_pso.py --dataset nasa_global

In [ ]:
# Step 16: NASA Global Multi-Temperature Comparison Table
!python benchmark_comparison.py --dataset nasa_global

---
## PART 4: Cross-Dataset Multi-Panel Performance Summary

In [ ]:
# Step 17: Cross-Dataset Multi-Panel Plots (Direct Display)
!python plot_cross_dataset.py